In [2]:
# seed_utils.py
import os
import random
import numpy as np

def seed_everything(seed: int = 42, deterministic: bool = True):
    """
    在对象定义与训练运行前调用：
    - 设定 Python/NumPy/PyTorch/igraph 的随机种子
    - 尽可能启用确定性计算
    """
    # 1) Python 层
    os.environ["PYTHONHASHSEED"] = str(seed)   # 影响哈希相关的随机性（如 dict 遍历顺序）
    random.seed(seed)

    # 2) NumPy
    np.random.seed(seed)

    # 3) PyTorch
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # 多卡

        # cudnn 配置
        if deterministic:
            # 为了完全可复现，禁用 cudnn 的自动算法搜索
            import torch.backends.cudnn as cudnn
            cudnn.deterministic = True
            cudnn.benchmark = False

            # 强制使用确定性算子（遇到不支持的算子会抛异常，便于你及时替换）
            torch.use_deterministic_algorithms(True, warn_only=False)

            # 对某些 CUDA 10.2+ 的场景（如 cuBLAS）再加一层保障
            # （注意：必须在进程启动前设置；部分环境下可能无需这行）
            os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
        else:
            # 更快但不完全可复现的设置
            import torch.backends.cudnn as cudnn
            cudnn.benchmark = True
    except Exception as e:
        print(f"[seed_everything] PyTorch seeding skipped: {e}")

    # 4) igraph（python-igraph）
    # 不同版本 API 略有差异，这里做了向后兼容的尝试：
    try:
        import igraph as ig
        try:
            # 0.10/0.11+ 推荐写法
            # ig.random.RandomState 是 igraph 的 RNG；设置为全局 RNG
            ig.set_random_number_generator(ig.random.RandomState(seed))
        except Exception:
            # 旧版本兜底（若有）
            # 有些旧版本可能是 igraph.set_random_number_generator 或 igraph.RandomState
            # 下面做一次通用回退，不报错即可：
            if hasattr(ig, "RandomState"):
                ig.set_random_number_generator(ig.RandomState(seed))
    except Exception as e:
        print(f"[seed_everything] igraph seeding skipped: {e}")

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
def get_metric(dmt_object,fate):
    cell_meta = dmt_object.tree_data.G.nodes['HSPC']['data'].adata.obs.copy()
    cell_meta['fate'] = dmt_object.tree_data.G.nodes['HSPC']['data'].adata.obs['Monocyte_fate']/(dmt_object.tree_data.G.nodes['HSPC']['data'].adata.obs['Monocyte_fate']+dmt_object.tree_data.G.nodes['HSPC']['data'].adata.obs['Neutrophil_fate'])
    cell_meta['fate1'] = pd.DataFrame(dmt_object.tree_data.tree['HSPC']['prior_list']['prior'])['Monocyte'].values/(pd.DataFrame(dmt_object.tree_data.tree['HSPC']['prior_list']['prior'])['Monocyte'].values+pd.DataFrame(dmt_object.tree_data.tree['HSPC']['prior_list']['prior'])['Neutrophil'].values)
    cell_meta['truth_label'] = np.where(cell_meta['Weinreb_fate'] > 0.5, 1, 0)
    cell_meta['predict_label'] = np.where(cell_meta[fate] > 0.5, 1, 0)

    f1 = f1_score(y_true=cell_meta['truth_label'].values, y_pred=cell_meta['predict_label'].values, average='macro')
    acc = accuracy_score(y_true=cell_meta['truth_label'].values, y_pred=cell_meta['predict_label'].values)
    auroc = roc_auc_score(cell_meta['truth_label'].values,cell_meta[fate].values)
    pearson = cell_meta['Weinreb_fate'].corr(cell_meta[fate])
    spearman = cell_meta['Weinreb_fate'].corr(cell_meta[fate],method='spearman')

    return [auroc,acc,f1,pearson,spearman]
def get_metric1(dmt_object,fate):
    cell_meta = dmt_object.tree_data.G.nodes['EE']['data'].adata.obs.copy()
    cell_meta['fate'] = dmt_object.tree_data.G.nodes['EE']['data'].adata.obs['TE_fate']/(dmt_object.tree_data.G.nodes['EE']['data'].adata.obs['TE_fate']+dmt_object.tree_data.G.nodes['EE']['data'].adata.obs['MP_fate'])
    cell_meta['fate1'] = pd.DataFrame(dmt_object.tree_data.tree['EE']['prior_list']['prior'])['TE'].values/(pd.DataFrame(dmt_object.tree_data.tree['EE']['prior_list']['prior'])['TE'].values+pd.DataFrame(dmt_object.tree_data.tree['EE']['prior_list']['prior'])['MP'].values)
    cell_meta['truth_label'] = np.where(cell_meta['fate_bias'] > 0.5, 1, 0)
    cell_meta['predict_label'] = np.where(cell_meta[fate] > 0.5, 1, 0)

    f1 = f1_score(y_true=cell_meta['truth_label'].values, y_pred=cell_meta['predict_label'].values, average='macro')
    acc = accuracy_score(y_true=cell_meta['truth_label'].values, y_pred=cell_meta['predict_label'].values)
    auroc = roc_auc_score(cell_meta['truth_label'].values,cell_meta[fate].values)
    pearson = cell_meta['fate_bias'].corr(cell_meta[fate])
    spearman = cell_meta['fate_bias'].corr(cell_meta[fate],method='spearman')

    return [auroc,acc,f1,pearson,spearman]

In [3]:
from model import DyMoTree
seed_everything(1, deterministic=True)

'''
 
dmt = DyMoTree(d_path='/data02/work/wangjiayi/Data/scRNA/CART_tju/',
                 task='CART',
                 n_neighbor=50,
                 device='cuda')   
dmt = DyMoTree(d_path='/data02/work/wangjiayi/Data/scLT/Weinreb/BiTree/desc246',
                 task='Larry',
                 n_neighbor=50,
                 device='cuda') 
dmt = DyMoTree(d_path='/data02/work/wangjiayi/Data/scRNA/CD8/truth',
                 task='CD8',
                 n_neighbor=50,
                 device='cuda')   
dmt = DyMoTree(d_path='/data02/work/wangjiayi/Project_Data/scRNA/mmEndoderm/Data.DyMoTree/',
                 task='mmE',
                 n_neighbor=40,
                 device='cuda')   
dmt = DyMoTree(d_path='/data02/work/wangjiayi/Project_Data/scRNA/LungCancer/Data.DyMoTree/',
                 task='LC',
                 n_neighbor=50,
                 device='cuda')   
       
'''
dmt = DyMoTree(d_path='/data02/work/wangjiayi/Data/scLT/Weinreb/BiTree/desc246',
                 task='Larry',
                 n_neighbor=50,
                 device='cuda') 

TypeError: DyMoTree.__init__() missing 1 required positional argument: 'epsilon'

In [8]:
import random
r = 0
run = 20
metric = pd.DataFrame(columns=['AUROC','Accuracy','F1-score','Pearson','Spearman'])
seeds = [392,375,426,192,410,474,426,628,488,808,777,289,306,1019,532,891,556,824,125,976]#[random.randint(0, 2**10 - 1) for _ in range(run)]
#[392,375,426,192,410,474,426,628,488,808,777,289,306,1019,532,891,556,824,125,976]
for i in seeds:
    seed_everything(i, deterministic=True)
    dmt.train(
            lamda1=0.5,lamda2=0.5,lamda3=0.1,lamda4=0,
            k=1e-4, # best: 0
            c=1,  # best: 0.1
            e=0,   # best: 10
            b=0, # best: 1e-4
            n_iter=200,
            pre_iter=200,
            lr=1e-4,
            pre_lr=1e-5)
    dmt.get_fate_sapce()
    metric.loc[r] = get_metric(dmt,'fate')
    r+=1

:: Do pre-train Graph encoder for HSPC::


100%|█████████████████████████████████████████████████| 100/100 [00:11<00:00,  9.02it/s, loss=0.635]


:: Do pre-train Graph encoder for Monocyte::


100%|█████████████████████████████████████████████████| 100/100 [00:07<00:00, 13.06it/s, loss=0.651]


:: Do pre-train Graph encoder for Neutrophil::


...

:: DyMoTree stage1 tranning ::


Iter: 200/200 : 100%|█████████████████████████████████| 200/200 [00:50<00:00,  3.93it/s, loss=0.781]


:: DyMoTree stage2 tranning ::


Iter: 200/200 : 100%|██████████████████████████████████| 200/200 [01:24<00:00,  2.35it/s, loss=1.25]


[2025-10-30 02:42:35] get fate space of HSPC


In [9]:
metric.mean()

AUROC       0.831108
Accuracy    0.737641
F1-score    0.737052
Pearson     0.577295
Spearman    0.585612
dtype: float64

In [10]:
get_metric(dmt,'fate1')

[0.7981851907907936,
 0.7163684559310802,
 0.7160181367927847,
 0.5048602154842101,
 0.537506883070152]

In [12]:
metric.to_csv('./Fig2/1.Larry.DMT.bench.result/performance.csv')

In [4]:
# hyperparameter test
# KNN number  | knn in [5,10,20,30,40,50]
import random
import pandas as pd
from model import DyMoTree

seeds = [392,375,426,192,410,474,426,628,488,808,777,289,306,1019,532,891,556,824,125,976]#[random.randint(0, 2**10 - 1) for _ in range(run)]
nn = [5,10,20,30,40,50]
#[392,375,426,192,410,474,426,628,488,808,777,289,306,1019,532,891,556,824,125,976]
metric = pd.DataFrame(columns=['param','run','AUROC','Accuracy','F1-score','Pearson','Spearman'])
r = 0
for n in nn:
        dmt = DyMoTree(d_path='/data02/work/wangjiayi/Data/scLT/Weinreb/BiTree/desc246',
                 task='Larry',
                 n_neighbor=n,
                 device='cuda') 
        for i in seeds:
                seed_everything(i, deterministic=True)
                dmt.train(
                        lamda1=0.5,lamda2=0.5,lamda3=0.1,lamda4=0,
                        k=1e-4, # best: 0
                        c=1,  # best: 0.1
                        e=0,   # best: 10
                        b=0, # best: 1e-4
                        n_iter=200,
                        pre_iter=200,
                        lr=1e-4,
                        pre_lr=1e-5)
                dmt.get_fate_sapce()
                res_ = [f'nn={n}', r] + get_metric(dmt, 'fate')
                metric.loc[r] = res_
                r+=1
metric.to_csv('./Fig2/1.Larry.DMT.bench.result/KNN_bench.csv')

[2025-10-30 10:27:29] loading node data for HSPC
[2025-10-30 10:27:30] loading node data for Monocyte
[2025-10-30 10:27:32] loading node data for Neutrophil
[2025-10-30 10:27:33] calculate shortest distance for HSPC with all descendant


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-1)]: Done  29 tasks      | elapsed:    0.8s
[Parallel(n_jobs=-1)]: Done  40 out of  64 | elapsed:    0.9s remaining:    0.5s
[Parallel(n_jobs=-1)]: Done  47 out of  64 | elapsed:    0.9s remaining:    0.3s
[Parallel(n_jobs=-1)]: Done  54 out of  64 | elapsed:    1.0s remaining:    0.2s
[Parallel(n_jobs=-1)]: Done  61 out of  64 | elapsed:    1.0s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  64 out of  64 | elapsed:    1.0s finished


[2025-10-30 10:27:35] loading edge data for HSPC
:: Do pre-train Graph encoder for HSPC::


100%|█████████████████████████████████████████████████| 100/100 [00:02<00:00, 48.30it/s, loss=0.715]


:: Do pre-train Graph encoder for Monocyte::


...

:: DyMoTree stage1 tranning ::


Iter: 200/200 : 100%|█████████████████████████████████| 200/200 [00:50<00:00,  3.92it/s, loss=0.781]


:: DyMoTree stage2 tranning ::


Iter: 200/200 : 100%|██████████████████████████████████| 200/200 [01:29<00:00,  2.24it/s, loss=1.25]


[2025-10-30 14:22:21] get fate space of HSPC


In [5]:
# hyperparameter test
# Cell Portion  | data in [desc2,desc4,desc6,desc246]
import random
import pandas as pd
from model import DyMoTree

seeds = [392,375,426,192,410,474,426,628,488,808,777,289,306,1019,532,891,556,824,125,976]#[random.randint(0, 2**10 - 1) for _ in range(run)]
#data = ['desc2','desc4','desc6','desc246']
data = ['desc6']
#[392,375,426,192,410,474,426,628,488,808,777,289,306,1019,532,891,556,824,125,976]
metric = pd.DataFrame(columns=['param','run','AUROC','Accuracy','F1-score','Pearson','Spearman'])
for d in data:
        r = 0
        run = 20
        dmt = DyMoTree(d_path=f'/data02/work/wangjiayi/Data/scLT/Weinreb/BiTree/{d}',
                 task='Larry',
                 n_neighbor=50,
                 device='cpu') 
        for i in seeds:
                seed_everything(i, deterministic=True)
                dmt.train(
                        lamda1=0.5,lamda2=0.5,lamda3=0.1,lamda4=0,
                        k=1e-4, # best: 0
                        c=1,  # best: 0.1
                        e=0,   # best: 10
                        b=0, # best: 1e-4
                        n_iter=200,
                        pre_iter=200,
                        lr=1e-4,
                        pre_lr=1e-5)
                dmt.get_fate_sapce()
                res_ = [f'Cell portion={d}', r] + get_metric(dmt, 'fate')
                metric.loc[r] = res_
                r+=1
metric.to_csv('./Fig2/1.Larry.DMT.bench.result/Cell_bench_desc6.csv')

[2025-10-30 14:39:13] loading node data for HSPC
[2025-10-30 14:39:14] loading node data for Monocyte
[2025-10-30 14:39:17] loading node data for Neutrophil
[2025-10-30 14:39:19] calculate shortest distance for HSPC with all descendant


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    6.9s
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:   12.6s
[Parallel(n_jobs=-1)]: Done  29 tasks      | elapsed:   13.4s
[Parallel(n_jobs=-1)]: Done  40 out of  64 | elapsed:   19.3s remaining:   11.6s
[Parallel(n_jobs=-1)]: Done  47 out of  64 | elapsed:   19.7s remaining:    7.1s
[Parallel(n_jobs=-1)]: Done  54 out of  64 | elapsed:   24.9s remaining:    4.6s
[Parallel(n_jobs=-1)]: Done  61 out of  64 | elapsed:   25.7s remaining:    1.3s
[Parallel(n_jobs=-1)]: Done  64 out of  64 | elapsed:   26.4s finished


[2025-10-30 14:39:51] loading edge data for HSPC
:: Do pre-train Graph encoder for HSPC::


100%|█████████████████████████████████████████████████| 100/100 [00:35<00:00,  2.84it/s, loss=0.633]


:: Do pre-train Graph encoder for Monocyte::


...

:: DyMoTree stage1 tranning ::


Iter: 200/200 : 100%|█████████████████████████████████| 200/200 [15:51<00:00,  4.76s/it, loss=0.807]


:: DyMoTree stage2 tranning ::


Iter: 200/200 : 100%|███████████████████████████████████| 200/200 [16:58<00:00,  5.09s/it, loss=1.5]


[2025-10-31 03:01:55] get fate space of HSPC


In [ ]:
metric.to_csv('./Fig2/1.Larry.DMT.bench.result/Cell_bench_desc6.csv')

In [ ]:
# hyperparameter test
# Wasserstein loss  | lambda3 in [0,0.01,0.1,0.5,1]
import random
import pandas as pd
from model import DyMoTree

seeds = [392,375,426,192,410,474,426,628,488,808,777,289,306,1019,532,891,556,824,125,976]#[random.randint(0, 2**10 - 1) for _ in range(run)]
lambda3 = [0,0.01,0.1,0.5,1]
#[392,375,426,192,410,474,426,628,488,808,777,289,306,1019,532,891,556,824,125,976]
metric = pd.DataFrame(columns=['param','run','AUROC','Accuracy','F1-score','Pearson','Spearman'])
for lam in lambda3:
        r = 0
        run = 20
        dmt = DyMoTree(d_path=f'/data02/work/wangjiayi/Data/scLT/Weinreb/BiTree/desc246',
                 task='Larry',
                 n_neighbor=50,
                 device='cuda') 
        for i in seeds:
                seed_everything(i, deterministic=True)
                dmt.train(
                        lamda1=0.5,lamda2=0.5,lamda3=lam,lamda4=0,
                        k=1e-4, # best: 0
                        c=1,  # best: 0.1
                        e=0,   # best: 10
                        b=0, # best: 1e-4
                        n_iter=200,
                        pre_iter=200,
                        lr=1e-4,
                        pre_lr=1e-5)
                dmt.get_fate_sapce()
                res_ = [f'wasserstein={lam}', r] + get_metric(dmt, 'fate')
                metric.loc[r] = res_
                r+=1
metric.to_csv('./Fig2/1.Larry.DMT.bench.result/Wasserstein_bench.csv')

In [ ]:
# hyperparameter test
# stage1 pretrain learning rate  | pre_lr in [1e-3,1e-5]